In [1]:
import csv
import random

# 定义CSV文件的列名
columns = ['Column1', 'Column2', 'Column3', 'Column4']

# 生成随机数据
data = []
for _ in range(10):  # 生成10行数据
    row = [
        random.randint(1, 100),  # 随机整数
        random.random(),  # 随机浮点数
        random.choice(['A', 'B', 'C']),  # 随机选择一个字符
        random.randint(1000, 9999)  # 随机整数
    ]
    data.append(row)

# 写入CSV文件
with open('data.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(columns)  # 写入列名
    writer.writerows(data)  # 写入数据

print("随机CSV文件已生成：random_data.csv")

随机CSV文件已生成：random_data.csv


In [ ]:
from langchain.document_loaders.csv_loader import CSVLoader

# 假设CSV文件名为data.csv
loader = CSVLoader(file_path='new.csv')
documents = loader.load()
print(documents)

In [2]:
from langchain.document_loaders.csv_loader import CSVLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

csv_files = {
    '新闻': ['new.csv', 'hacker.csv'],
}

all_documents = []
for category, file_paths in csv_files.items():
    for file_path in file_paths:
        try:
            loader = CSVLoader(file_path=file_path, encoding='utf-8')
            documents = loader.load()
            for doc in documents:
                doc.metadata['category'] = category  # 添加类别标签
            all_documents.extend(documents)
        except Exception as e:
            try:
                loader = CSVLoader(file_path=file_path, encoding='gbk')
                documents = loader.load()
                for doc in documents:
                    doc.metadata['category'] = category  # 添加类别标签
                all_documents.extend(documents)
            except Exception as e:
                print(f"Error loading {file_path}: {e}")

In [3]:
# 分割文档
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
split_documents = text_splitter.split_documents(all_documents)

print(len(split_documents))

# 打印分割后的文档
# for doc in split_documents:
#     print(doc)

841


In [ ]:
%pip install faiss-cpu

In [4]:
from langchain.vectorstores.faiss import FAISS
from langchain.embeddings.openai import OpenAIEmbeddings

# 创建嵌入
embeddings = OpenAIEmbeddings(base_url='https://api.chatanywhere.tech/v1/',api_key='sk-dw366zNMWe7tTmmRfzr0NTVMCjegXOCTU1PRdT3wTFvHjr4X',model='text-embedding-ada-002')

# 创建向量存储
vectorstore = FAISS.from_documents(split_documents, embeddings)

vectorstore.save_local("faiss_index")

In [7]:
from langchain.chains.retrieval_qa.base import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# 创建自定义提示模板
template = """
从现在开始你是Bug页面的网站向导，Bug网站网址http://cy.azyasaxi.cloudns.org/.

对于技术问题，请详细解释并提供相关链接。
对于一般问题，请简洁明了地回答。
对于导航问题，请提供清晰的路径和步骤。

然后当有用户向你提问{question}的时候,你需要从{context}里面抽取结果回答用户的{question},无法获取到相关信息回答用户的，则只需要跟用户强调自己的能力有限，无法回答这种问题
,给用户的回复会带上对于的网站链接
question: {question}
"""
prompt = PromptTemplate(template=template, input_variables=["question", "context"])

# 创建问答链
# https://api.chatanywhere.tech/v1/ sk-J5SBZ1hgnvUwJ4tfclSiET53pGaxP3m7mnIKfy3eoFXOGOEt
qa_chain = RetrievalQA.from_chain_type(
    llm=ChatOpenAI(temperature=0, base_url='http://127.0.0.1:8000/v1/', api_key='c739981ee79541deb8414f0bd8bb576c'),
    chain_type="stuff",
    # retriever=vectorstore.as_retriever(search_type="mmr",search_kwargs={'k': 10}),
    retriever=vectorstore.as_retriever(search_type="similarity_score_threshold",search_kwargs={"score_threshold": 0.5}),
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt}
)

#有没有网站安全测试的 ChatGPT的
# 进行问答
question = "有没有2024年六月的新闻呢？"
result = qa_chain({"query": question})

# 打印检索到的每个文档的标题
print("检索到的标题:")
print(result['source_documents'])
print("\n\n",result['result'])


检索到的标题:
[Document(page_content='id: a0ddeda4-281e-4caa-965a-dc544d747854\ndate: 20240506\ncategory: 知识库精选\ntitle: 知识库精选- 5 月 6 日\nlastEditedDate: 2024-05-07T00:54:00.035Z\nimage: https://assets.waytoagi.com/usercontent/Xnapper_2024_05_06_23_04_32_9965169791.png?t=a0ddeda4-281e-4caa-965a-dc544d747854&image_process=resize,w_506\nurl: https://blog.waytoagi.com/article/news-20240506\nsummary: 1. AI Talk 汗青老师的直播分享\n2. OpenAI Sam Altman最新对话MIT校长 2024.5\n3. 2024Q2 再谈 Agent 构建平台的产品设计\n4. 2024 年巴菲特股东大会 4 万字全文：AI 的影响力，堪比原子弹！', metadata={'source': 'new.csv', 'row': 45, 'category': '新闻'}), Document(page_content='id: 87b99b9e-baab-4ec9-abbd-d69de3bd7d28\ndate: 20240407\ncategory: 知识库精选\ntitle: 知识库精选- 4 月 7 日\nlastEditedDate: 2024-04-09T01:10:01.297Z\nimage: https://assets.waytoagi.com/usercontent/Xnapper_2024_04_07_11_21_01_fe99480bbf.png?t=87b99b9e-baab-4ec9-abbd-d69de3bd7d28&image_process=resize,w_506\nurl: https://blog.waytoagi.com/article/news-20240407\nsummary: 1. 《智象&凯正咨询：2024中国新科技出海报告》\n2. 《

In [ ]:
async for event in qa_chain.astream_events({"query": question},version="v1"):
    print(event)


In [ ]:
from langchain.document_loaders.csv_loader import CSVLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores.faiss import FAISS
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.chains.retrieval_qa.base import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain.chains.llm import LLMChain

csv_files = {
    '新闻': ['new.csv', 'hacker.csv'],
}

# 加载CSV文件并添加类别标签
def get_csv_to_vectorstore(csv_files: dict):
    all_documents = []
    for category, file_paths in csv_files.items():
        for file_path in file_paths:
            try:
                loader = CSVLoader(file_path=file_path, encoding='utf-8')
                documents = loader.load()
                for doc in documents:
                    doc.metadata['category'] = category  # 添加类别标签
                all_documents.extend(documents)
            except Exception as e:
                try:
                    loader = CSVLoader(file_path=file_path, encoding='gbk')
                    documents = loader.load()
                    for doc in documents:
                        doc.metadata['category'] = category  # 添加类别标签
                    all_documents.extend(documents)
                except Exception as e:
                    print(f"Error loading {file_path}: {e}")

    # 分割文档
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    split_documents = text_splitter.split_documents(all_documents)
    return split_documents

ack_knowledge = get_csv_to_vectorstore(csv_files)

def create_db(ack_knowledge: list):
    # 创建嵌入
    embeddings = OpenAIEmbeddings(base_url='https://api.chatanywhere.cn/v1/', api_key='sk-dw366zNMWe7tTmmRfzr0NTVMCjegXOCTU1PRdT3wTFvHjr4X', model='text-embedding-3-small')

    # 创建向量存储
    vectorstore = FAISS.from_documents(ack_knowledge, embeddings)
    vectorstore.save_local("faiss_index")
    return vectorstore

# 创建自定义提示模板
template = """
从现在开始你是用户们的网站向导，当有用户问“你是谁”或者“识图让你介绍自己”的时候，你都需要强调回答：“我是Bug页面的网站向导，基于GPT4模型，可以为您正确的了解Bug网站，这是我们的网站首页网址http://cy.azyasaxi.cloudns.org/”.
自此结束.

对于技术问题，请详细解释并提供相关链接。
对于一般问题，请简洁明了地回答。
对于导航问题，请提供清晰的路径和步骤。

当用户表达感谢时，请回复“不客气，很高兴能帮到您！”。
当用户表达不满时，请回复“很抱歉给您带来不便，我会尽力改进。”。

然后当有用户向你提问{question}的时候,你需要从context: {context}里面抽取结果回答用户的{question},如果context为空或者是无法从context抽取答案回答用户的，则只需要跟用户强调自己的能力有限，无法回答这种问题.
记住先记得总结context在回复用户,给用户的回复会带上对于的网站链接
question: {question}
AI:
"""
prompt = PromptTemplate(template=template, input_variables=["question", "context"])

# 创建检索链
retrieval_chain = RetrievalQA.from_chain_type(
    llm=ChatOpenAI(temperature=0, base_url='https://api.chatanywhere.tech/v1/', api_key='sk-J5SBZ1hgnvUwJ4tfclSiET53pGaxP3m7mnIKfy3eoFXOGOEt'),
    chain_type="stuff",
    retriever=create_db(ack_knowledge).as_retriever(search_type="mmr", search_kwargs={'k': 10}),
    return_source_documents=True,
)

# 创建生成链
generation_chain = LLMChain(
    llm=ChatOpenAI(temperature=0, base_url='https://api.chatanywhere.tech/v1/', api_key='sk-J5SBZ1hgnvUwJ4tfclSiET53pGaxP3m7mnIKfy3eoFXOGOEt'),
    prompt=prompt,
)

def rag_csv_bot(query):
    # 使用检索链获取相关文档
    retrieval_result = retrieval_chain({"query": query})
    context = "\n".join([doc.page_content for doc in retrieval_result["source_documents"]])

    # 使用生成链生成最终回答
    generation_result = generation_chain({"question": query, "context": context})
    return generation_result["text"]

# 示例调用
response = rag_csv_bot("介绍一下Bug网站")
print(response)